# Session 2: fits, limits and the worst-case stack
**A. Real-world problem.** A 25 mm shaft must turn freely in a bearing bore, but a gear must
stay put on another 25 mm shaft. Both are "a shaft in a hole"; only the fit differs. The ISO
system names fits with a code such as 25 H7/g6. Today you decode it, compute the smallest and
largest clearance, pick fits for two jobs, and add up tolerances along a chain of parts.

*Educational model.* The table values below are the ISO 286 values for the size range
18 to 30 mm as we state them here; always check your own copy of the standard before you
design a real part.

In [ ]:
import itertools
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

# International tolerance grades for nominal sizes over 18 up to 30 mm, in micrometres (um)
IT = {6: 13.0, 7: 21.0}
# Shaft fundamental deviations for the same size range, in um.
# g and h: the UPPER deviation es is given (g: -7, h: 0). p: the LOWER deviation ei is given (+22).
# js is symmetric: +-IT/2.
SHAFT_DEV = {"g": ("es", -7.0), "h": ("es", 0.0), "p": ("ei", 22.0)}

## Step 1: tolerance zones in the hole-basis system
Deviations are measured from the nominal size (25 mm here) in micrometres (1 um = 0.001 mm).
**Hole basis:** the hole is always H, meaning its lower deviation EI is 0, so the hole zone is
`[0, +IT]`. A shaft letter moves the shaft zone: `[ei, es]` with `es - ei = IT`.

In [ ]:
def hole_zone(letter="H", grade=7):
    """(EI, ES) of the hole in um. Only the hole-basis hole H is supported."""
    if letter != "H":
        raise ValueError("hole-basis system: the hole letter is H")
    return 0.0, IT[grade]


def shaft_zone(letter, grade):
    """(ei, es) of the shaft in um."""
    it = IT[grade]
    if letter == "js":
        return -it / 2, it / 2
    kind, value = SHAFT_DEV[letter]
    if kind == "es":
        return value - it, value
    return value, value + it


def to_mm(nominal_mm, zone_um):
    """Convert a zone in um to limits in mm: nominal + deviation / 1000."""
    return tuple(round(nominal_mm + z / 1000.0, 6) for z in zone_um)

## Step 2: clearance limits of a fit
    minimum clearance = EI - es       (smallest hole, biggest shaft)
    maximum clearance = ES - ei       (biggest hole, smallest shaft)
Negative means interference. The fit is a **clearance fit** if the minimum is >= 0, an
**interference fit** if the maximum is <= 0, otherwise a **transition fit** (either can occur).

In [ ]:
def fit_limits(hole, shaft):
    """(min clearance, max clearance) in um from hole=(EI, ES) and shaft=(ei, es)."""
    return hole[0] - shaft[1], hole[1] - shaft[0]


def classify(min_c, max_c):
    if min_c >= 0:
        return "clearance"
    if max_c <= 0:
        return "interference"
    return "transition"


def fit(shaft_letter, shaft_grade, hole_grade=7):
    h = hole_zone("H", hole_grade)
    s = shaft_zone(shaft_letter, shaft_grade)
    lo, hi = fit_limits(h, s)
    return lo, hi, classify(lo, hi)

## Step 3: choose a fit for a job (design problem)
A job gives requirements on the clearance range. We test candidate fits against them.
*Sliding bearing (illustrative requirement):* at least 5 um of clearance for an oil film, at
most 45 um so it does not rattle. *Light press fit (illustrative):* interference in every
build, at most 40 um so the hole is not over-stressed.

In [ ]:
CANDIDATES = [("g", 6), ("h", 6), ("js", 6), ("p", 6)]


def choose_fits(min_c_required, max_c_allowed, hole_grade=7):
    """Names of candidate fits whose whole clearance range lies inside [min_c_required, max_c_allowed]."""
    out = []
    for letter, grade in CANDIDATES:
        lo, hi, _ = fit(letter, grade, hole_grade)
        if lo >= min_c_required and hi <= max_c_allowed:
            out.append(f"H{hole_grade}/{letter}{grade}")
    return out

## Step 4: worst-case tolerance stack
A gap is made by a chain of parts: lengths that add (sign +1) and lengths that subtract (-1):

    gap = sum( sign_i * x_i ),  each x_i = nominal_i +- t_i

In the worst case every part sits at its limit in the direction that hurts, so
**the tolerances simply add**: gap tolerance = sum of t_i. We check this by trying all
2^n combinations of upper/lower limits (the "corners").

In [ ]:
def worst_case_stack(signs, nominals, tols):
    """(nominal gap, lowest gap, highest gap) by adding the tolerances."""
    if any(t < 0 for t in tols):
        raise ValueError("tolerances must not be negative")
    nominal = float(np.dot(signs, nominals))
    total = float(np.sum(tols))
    return nominal, nominal - total, nominal + total


def corner_stack(signs, nominals, tols):
    """(lowest gap, highest gap) by trying every corner: each part at nominal - t or nominal + t."""
    gaps = []
    for pick in itertools.product((-1, 1), repeat=len(nominals)):
        x = np.asarray(nominals) + np.asarray(pick) * np.asarray(tols)
        gaps.append(float(np.dot(signs, x)))
    return min(gaps), max(gaps)

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    h = hole_zone("H", 7)
    fig, axes = plt.subplots(1, 3, figsize=(9, 4), sharey=True)
    for ax, (letter, grade) in zip(axes, [("g", 6), ("js", 6), ("p", 6)]):
        s = shaft_zone(letter, grade)
        lo, hi, kind = fit(letter, grade)
        ax.axhline(0, color="k", lw=1)
        ax.bar(-0.2, h[1] - h[0], bottom=h[0], width=0.35, color="tab:blue", alpha=0.7, label="hole H7")
        ax.bar(0.2, s[1] - s[0], bottom=s[0], width=0.35, color="tab:orange", alpha=0.8, label="shaft")
        ax.set_title(f"25 H7/{letter}{grade}\n{kind}: {lo:g} to {hi:g} um", fontsize=9)
        ax.set_xticks([])
        ax.grid(alpha=0.3, axis="y")
        print(f"25 H7/{letter}{grade}: hole {to_mm(25, h)} mm, shaft {to_mm(25, s)} mm, clearance {lo:g} to {hi:g} um ({kind})")
    axes[0].set_ylabel("deviation from 25 mm (um)")
    axes[0].legend(fontsize=7, loc="upper left")
    fig.tight_layout()
    fig.savefig(OUT / "fit_zones.png", dpi=120)

    signs = np.array([1, -1, -1, -1])
    nominals = np.array([40.0, 10.0, 12.0, 15.0])
    tols = np.array([0.10, 0.05, 0.05, 0.08])
    nom, lo, hi = worst_case_stack(signs, nominals, tols)
    clo, chi = corner_stack(signs, nominals, tols)
    print(f"stack: nominal gap {nom:.2f} mm, worst case {lo:.2f} to {hi:.2f} mm, corners {clo:.2f} to {chi:.2f} mm")
    print("sliding bearing candidates:", choose_fits(5, 45))
    print("light press candidates:", choose_fits(-40, -0.0001))

    fig2, ax2 = plt.subplots(figsize=(6.5, 3.6))
    names = ["housing", "plate 1", "plate 2", "plate 3"]
    ax2.barh(names, tols, color=["tab:blue", "tab:orange", "tab:orange", "tab:orange"])
    ax2.barh(["gap (sum)"], [tols.sum()], color="tab:green")
    ax2.set_xlabel("tolerance, plus or minus (mm)")
    ax2.set_title("Worst case: the gap tolerance is the sum of the part tolerances")
    ax2.grid(alpha=0.3, axis="x")
    fig2.tight_layout()
    fig2.savefig(OUT / "stack_sum.png", dpi=120)

## Where this stops
Adding every tolerance is safe but pessimistic: it assumes every part is at its worst limit at
the same time. Session 3 asks how likely that is.

## Coding tasks
1. **Easy.** Use `fit("h", 6)` and `fit("js", 6)` in `lesson.py` and print the clearance limits; predict them by hand first.
2. **Stretch.** Add a new row to the stack: a fourth plate of 5.00 +-0.03 mm that also subtracts. Print the new nominal gap and worst-case limits and check them with `corner_stack`.
3. **Explain.** Explain why a part tolerance that is symmetric, plus or minus t, contributes t to the gap tolerance whether it adds or subtracts in the chain.

## Assessment
- **A. Conceptual.** Why is a fit with minimum clearance 0 (such as H7/h6) still called a clearance fit, and what separates a transition fit from an interference fit?
- **B. Numerical.** Find the smallest and largest clearance of 25 H7/h6, using IT6 = 13 um, IT7 = 21 um and es = 0 for h. Sanity check: what must their difference be?
- **C. Graph and simulation interpretation.** In the zone figure, how far above the top of the H7 hole zone does the p6 shaft zone start? What does that mean for assembly?
- **D. Design and optimisation challenge.** Design: select a fit from H7/g6, H7/h6, H7/js6 and H7/p6 for (a) a sliding bearing that needs at least 5 um and at most 45 um of clearance and (b) a gear that must stay on its shaft with interference in every build and at most 40 um of it. Give the clearance limits that justify your choice, and say what is wrong with the others.

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    checks.append(("hole H7 zone is 0 to +21 um", L.hole_zone("H", 7) == (0.0, 21.0)))
    ei, es = L.shaft_zone("g", 6)
    checks.append(("shaft g6 zone is -20 to -7 um", (ei, es) == (-20.0, -7.0)))
    checks.append(("shaft zone width equals the IT6 grade, 13 um", abs((es - ei) - 13.0) < 1e-9))
    checks.append(("js6 is symmetric, +-6.5 um", L.shaft_zone("js", 6) == (-6.5, 6.5)))
    checks.append(("p6 zone is +22 to +35 um", L.shaft_zone("p", 6) == (22.0, 35.0)))

    lo, hi, kind = L.fit("g", 6)
    checks.append((f"worked example: 25 H7/g6 clearance is 7 to 41 um, a clearance fit (got {lo:g} to {hi:g}, {kind})",
                   lo == 7.0 and hi == 41.0 and kind == "clearance"))
    hmm = L.to_mm(25, L.hole_zone("H", 7))
    smm = L.to_mm(25, L.shaft_zone("g", 6))
    checks.append(("limits in mm: hole 25.000 to 25.021, shaft 24.980 to 24.993",
                   hmm == (25.0, 25.021) and smm == (24.98, 24.993)))
    # hand calculation: the spread of the fit equals the sum of the two tolerances, 21 + 13 = 34 um
    checks.append(("hand calculation: max - min clearance = IT7 + IT6 = 34 um", abs((hi - lo) - 34.0) < 1e-9))
    # the same numbers recomputed from mm limits
    mn = round((hmm[0] - smm[1]) * 1000, 6)
    mx = round((hmm[1] - smm[0]) * 1000, 6)
    checks.append(("clearance recomputed from the mm limits is 7 and 41 um", mn == 7.0 and mx == 41.0))

    lo, hi, kind = L.fit("h", 6)
    checks.append(("H7/h6 is a clearance fit from 0 to 34 um (zero minimum still counts as clearance)", (lo, hi, kind) == (0.0, 34.0, "clearance")))
    lo, hi, kind = L.fit("js", 6)
    checks.append(("H7/js6 is a transition fit, -6.5 to +27.5 um", (lo, hi, kind) == (-6.5, 27.5, "transition")))
    lo, hi, kind = L.fit("p", 6)
    checks.append(("H7/p6 is an interference fit, max clearance -1 and min clearance -35 um", (lo, hi, kind) == (-35.0, -1.0, "interference")))
    try:
        L.hole_zone("G", 7)
        bad = False
    except ValueError:
        bad = True
    checks.append(("edge: only the hole H is accepted in the hole-basis system", bad))

    checks.append(("design: only H7/g6 suits the sliding bearing (5 to 45 um)", L.choose_fits(5, 45) == ["H7/g6"]))
    checks.append(("design: only H7/p6 suits the light press (interference, at most 40 um)", L.choose_fits(-40, -0.0001) == ["H7/p6"]))
    checks.append(("design edge: an impossible requirement returns no fit", L.choose_fits(100, 120) == []))

    signs = np.array([1, -1, -1, -1])
    nominals = np.array([40.0, 10.0, 12.0, 15.0])
    tols = np.array([0.10, 0.05, 0.05, 0.08])
    nom, lo, hi = L.worst_case_stack(signs, nominals, tols)
    checks.append((f"worked example: stack gap 3.00 mm, worst case 2.72 to 3.28 (got {nom:.2f}, {lo:.2f} to {hi:.2f})",
                   abs(nom - 3.0) < 1e-9 and abs(lo - 2.72) < 1e-9 and abs(hi - 3.28) < 1e-9))
    clo, chi = L.corner_stack(signs, nominals, tols)
    checks.append(("exhaustive corners (16 combinations) agree with the sum of tolerances", abs(clo - lo) < 1e-9 and abs(chi - hi) < 1e-9))
    rng = np.random.default_rng(1)
    ok = True
    for _ in range(20):
        n = int(rng.integers(2, 7))
        s = rng.choice([-1, 1], n)
        x = rng.uniform(5, 50, n)
        t = rng.uniform(0.01, 0.2, n)
        a = L.worst_case_stack(s, x, t)
        b = L.corner_stack(s, x, t)
        ok &= abs(a[1] - b[0]) < 1e-9 and abs(a[2] - b[1]) < 1e-9
    checks.append(("20 random stacks: sum of tolerances equals the corner enumeration", bool(ok)))
    checks.append(("edge: zero tolerances give a single gap", L.worst_case_stack(signs, nominals, np.zeros(4)) == (3.0, 3.0, 3.0)))
    try:
        L.worst_case_stack(signs, nominals, [-0.1, 0, 0, 0])
        bad = False
    except ValueError:
        bad = True
    checks.append(("edge: a negative tolerance is rejected", bad))
    # numbers used in the text and assessment
    checks.append(("stack tolerances add to 0.28 mm, so the gap window is 0.56 mm wide", abs(tols.sum() - 0.28) < 1e-9 and abs((hi - lo) - 0.56) < 1e-9))
    w = L.worst_case_stack(signs, nominals, np.full(4, 0.025))
    checks.append(("four parts of +-0.025 mm keep the worst-case gap at 2.90 mm or more (and 3.10 or less)",
                   abs(w[1] - 2.90) < 1e-9 and abs(w[2] - 3.10) < 1e-9))
    checks.append(("a hole-basis shaft p6 starts 1 um above the top of the H7 hole", L.shaft_zone("p", 6)[0] - L.hole_zone("H", 7)[1] == 1.0))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
